In [1]:
import pandas as pd

In [ ]:
df1 = pd.read_csv('dataset/olympics_medals_by_sport_year.csv')
print(df1.head())

       Sport  Year  Bronze  Gold  Silver  Total
0  Athletics  1896      12    12      13     37
1  Athletics  1900      22    27      27     76
2  Athletics  1904      23    28      28     79
3  Athletics  1906      21    21      23     65
4  Athletics  1908      33    34      34    101


# Data Cleaning and Preprocessing

In [6]:
# Check for missing values
missing_data = df1.isnull().sum()
print('Missing values:\n',missing_data)

# Check for unique values in "Medal" column (in case there are variations in how it is written)
unique_medals = df1['Medal'].unique()
print('\nUnique medals: ',unique_medals)

# Convert columns to appropriate types
df1['Year'] = df1['Year'].astype(int)
df1['Medal'] = df1['Medal'].str.strip()  # Remove extra spaces from medal names

# Handle missing values (e.g., 'No medal' for missing medals)
# df1['Medal'].fillna('No medal', inplace=True)
df_clean = df1.fillna({'Medal': 'No medal'}, inplace=True)
print(df_clean)

# Check the cleaned data
df1.head()

Missing values:
 player_id    0
Name         0
Sex          0
Team         0
NOC          0
Year         0
Season       0
City         0
Sport        0
Event        0
Medal        0
dtype: int64

Unique medals:  ['No medal' 'Gold' 'Bronze' 'Silver']
None


,player_id,Name,Sex,Team,NOC,Year,Season,City,Sport,Event,Medal
0,0,A Dijiang,M,China,CHN,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,No medal
1,1,A Lamusi,M,China,CHN,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,No medal
2,2,Gunnar Aaby,M,Denmark,DEN,1920,Summer,Antwerpen,Football,Football Men's Football,No medal
3,3,Edgar Aabye,M,Denmark/Sweden,DEN,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,26,Cornelia (-strannood),F,Netherlands,NED,1932,Summer,Los Angeles,Athletics,Athletics Women's 100 metres,No medal


# Research Questions
Q1. Analyze the development of teams' performances over time. Are there detectable trends in the performance of teams over the last decade?

In [12]:
# Filter data for the last decade (2014-2024)
last_decade_data = df1[(df1['Year'] >= 2014) & (df1['Year'] <= 2024)]

# Group by Team (Country) and Year to get medal counts
team_performance = last_decade_data.groupby(['Year', 'Team', 'Medal']).size().unstack(fill_value=0)

# Display the top few rows to understand the performance trend over time
team_performance.head()

Medal                Bronze  Gold  No medal  Silver
Year Team                                          
2016 Afghanistan          0     0         3       0
     Albania              0     0         6       0
     Algeria              0     0        72       2
     American Samoa       0     0         4       0
     Andorra              0     0         4       0

Identifying top countries with highest no. of medals

In [25]:
# Calculate the total medal count for each country over the last decade
total_medals_per_country = last_decade_data.groupby('Team')['Medal'].value_counts().unstack(fill_value=0)
total_medals_per_country['Total'] = total_medals_per_country.sum(axis=1)

# Sort countries by the total number of medals to get the top countries
top_countries = total_medals_per_country.sort_values('Total', ascending=False).head(10)

# Display the top 10 countries by total medal count
top_countries[['Gold', 'Silver', 'Bronze', 'Total']]

Medal,Gold,Silver,Bronze,Total
Team,,,,
United States,381,258,236,2409
France,138,192,94,1843
Japan,112,85,80,1821
Australia,92,106,136,1797
Germany,89,118,139,1772
China,174,136,108,1644
Great Britain,144,139,159,1551
Italy,61,81,86,1529
Canada,54,44,113,1444


Q2. How does the performance in different sports vary across decades or Olympic Games?

In [23]:
# Group by Sport and Year to get the medal counts
sport_performance_by_year = df1.groupby(['Year', 'Sport', 'Medal']).size().unstack(fill_value=0)

# Display the top few rows to see the trends in sports over time
sport_performance_by_year.head()

Medal            Bronze  Gold  No medal  Silver
Year Sport                                     
1896 Athletics       12    12        69      13
     Cycling          4     6        25       6
     Fencing          3     3         6       3
     Gymnastics       5    26        60       6
     Shooting         5     5        50       5

In [26]:
# Filter data for specific sports (Athletics, Swimming, Football) to analyze trends
sports_of_interest = ['Athletics', 'Swimming', 'Football']
sports_performance = df1[df1['Sport'].isin(sports_of_interest)]

# Group by Sport and Year to get the medal counts for these sports
sports_performance_by_year = sports_performance.groupby(['Year', 'Sport', 'Medal']).size().unstack(fill_value=0)

# Display the first few rows of the filtered and grouped data
sports_performance_by_year.head()

Medal           Bronze  Gold  No medal  Silver
Year Sport                                    
1896 Athletics      12    12        69      13
     Swimming        2     4         8       4
1900 Athletics      22    27       158      27
     Football       11    11         0      13
     Swimming       10    10       100      11

In [15]:
df1 = df1[(df1['Year'] >= 1896) & (df1['Year'] <= 2024)]

# Group the data by Sport and Medal type to get the count of each medal type (Gold, Silver, Bronze)
medal_counts = df1.groupby(['Sport', 'Medal']).size().unstack(fill_value=0)

# Add a column for total medals by summing Gold, Silver, and Bronze
medal_counts['Total'] = medal_counts['Gold'] + medal_counts['Silver'] + medal_counts['Bronze']

# Sort by Total medals to get the top 10 sports
top_10_sports = medal_counts.sort_values(by='Total', ascending=False).head(10)

# Reset index for better readability
top_10_sports.reset_index(inplace=True)

# Display the top 10 sports with medal counts
print(top_10_sports)

Medal       Sport  Bronze  Gold  No medal  Silver  Total
0       Athletics    1452  1500     38865    1477   4429
1        Swimming    1096  1241     22946    1133   3470
2          Rowing    1086  1074      8392    1073   3233
3      Gymnastics     719   791     24451     746   2256
4         Fencing     626   654      9635     643   1923
5        Football     626   601      6079     600   1827
6          Hockey     580   589      4526     569   1738
7       Wrestling     540   449      6294     451   1440
8        Shooting     443   446     11244     447   1336
9         Sailing     396   479      5947     444   1319


In [ ]:
# List of specific sports for the heatmap
specific_sports = ['Athletics', 'Cycling', 'Fencing', 'Football', 'Gymnastics', 
                   'Rowing', 'Sailing', 'Shooting', 'Swimming', 'Wrestling']

# Filter the dataset for the selected sports
filtered_data = df1[df1['Sport'].isin(specific_sports)]

# Filter out "No medal" entries and keep only Gold, Silver, and Bronze
filtered_data = filtered_data[filtered_data['Medal'].isin(['Gold', 'Silver', 'Bronze'])]

# Group the data by Sport, Year, and Medal type to get the count of each medal type per sport and year
medals_by_sport_year = filtered_data.groupby(['Sport', 'Year', 'Medal']).size().unstack(fill_value=0)

# Reset index to make it easier to save to CSV
medals_by_sport_year.reset_index(inplace=True)

# Add a column for the total number of medals per sport per year (Gold + Silver + Bronze)
medals_by_sport_year['Total'] = medals_by_sport_year['Gold'] + medals_by_sport_year['Silver'] + medals_by_sport_year['Bronze']

# Save the resulting data to a new CSV file
output_file_path = 'dataset/medals_by_sport_year.csv'
medals_by_sport_year.to_csv(output_file_path, index=False)

# Output the path to the saved CSV file
print(f"The medals by sport and year data has been saved to: {output_file_path}")


The medals by sport and year data has been saved to: olympics_medals_by_sport_year.csv


Q3. What is the performance of athletes based on gender? Are there any gender disparities in medal wins?

In [27]:
# Group data by Sex and Medal to analyze the distribution of medals by gender
gender_performance = df1.groupby(['Sex', 'Medal']).size().unstack(fill_value=0)

# Display the gender-based performance
gender_performance

Medal,Bronze,Gold,No medal,Silver
Sex,,,,
F,3992,3914,62224,3891
M,9078,9088,151523,8855
